In [2]:
# RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_ollama.embeddings import OllamaEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('groq_api_key')

# List of URLs to load documents from
urls = [
    'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/',
    'https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/'
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0)

# Split the documents into chunks
docs_splits = text_splitter.split_documents(docs_list)

# LLM
llm = ChatGroq(model='openai/gpt-oss-120b', max_tokens=300, temperature=0)

# Embedding
embedding = OllamaEmbeddings(model='nomic-embed-text')

# Add the document chunks to the 'vector store' using Groq
vectorstore = InMemoryVectorStore.from_documents(documents=docs_splits, embedding=embedding)
retriever = vectorstore.as_retriever(k=6)

In [3]:
from langsmith import traceable

# Add decorator
@traceable
def rag_bot(question: str) -> dict:
    # Relevant context
    docs = retriever.invoke(question)
    docs_string = ' '.join(doc.page_content for doc in docs)
    
    instruction = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
        Documents: {docs_string}"""
        
    ai_msg = llm.invoke([
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': question}
    ])
    
    return {'answer': ai_msg.content, 'documents': docs}

In [4]:
rag_bot('what is agents?')

{'answer': '**Agents** (in the context of LLM‑powered autonomous systems) are software entities that can act **autonomously** to accomplish tasks.  \nThey combine a large language model (LLM) – which serves as the “brain” for reasoning, understanding language, and generating plans – with a set of supporting components that give the system structure, memory, and the ability to interact with the outside world.\n\n### Core Characteristics of an Agent\n\n| Characteristic | What it means for the agent |\n|----------------|-----------------------------|\n| **Autonomy** | Once given a goal, the agent can decide on its own what steps to take, without needing a human to micromanage each action. |\n| **Goal‑oriented** | The agent receives a high‑level objective (e.g., “write a market‑analysis report”) and works toward achieving it. |\n| **Reasoning & Planning** | Using the LLM, the agent breaks the goal into **subgoals** and creates a **plan** that orders the necessary steps. |\n| **Execution',


In [ ]:
# Dataset

from langsmith import Client

import os
from dotenv import load_dotenv

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('groq_api_key')
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_api_key')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

client = Client()

# Define the examples for the dataset
examples = [
    {    
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

In [10]:
# Create the dataset and example in LangSmith
dataset_name = 'RAG Evaluation'
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['10463db8-4f3c-47b9-a9f2-87f524d9245f',
  '8d615f10-16d7-4da1-b07f-c2263236d02e',
  'ff7c3cda-9bed-44af-bffd-a6f5290ede67'],
 'count': 3,
 'as_of': '2026-04-20T15:54:01.342831473Z'}

### Evaluators or Metrics

1. Correctness: Response vs Reference answer

- `Goal`: Measure 'how similar/correct is the RAG chain answer, relative to a ground-truth answer'
- `Mode`: Requires a ground truth (reference) answer supplied through a dataset
- `Evaluator`: Use LLM-as-Judge to assess answer correctness

In [ ]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


grader_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0).with_structured_output(
    CorrectnessGrade,
    method="json_schema",
    strict=True
)

## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [12]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0).with_structured_output(
    RelevanceGrade,
    method="json_schema",
    strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [13]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0).with_structured_output(
    GroundedGrade,
    method="json_schema",
    strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [14]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0).with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
    strict=True
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Run the evaluation

In [16]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-357a396e' at:
https://smith.langchain.com/o/a1921ab0-b36c-4cd2-97ec-4f0a042f060c/datasets/59f78f13-af3d-42fb-b4b2-ec115df791f0/compare?selectedSessions=b79f6899-f099-4609-8e5d-04ebbe36cac7




3it [00:27,  9.15s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are five types of adversarial attacks?,,[page_content='There are various means to find...,None,Five types of adversarial attacks are (1) Toke...,False,False,False,True,0.987152,10463db8-4f3c-47b9-a9f2-87f524d9245f,019daba2-e377-7051-b9ce-8c93a36c9ca3
1,How does the ReAct agent use self-reflection?,**Short answer:** \nThe ReAct (Reason + Act) ...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",False,True,True,True,0.892152,8d615f10-16d7-4da1-b07f-c2263236d02e,019daba2-f7a5-7553-9006-997ad8e04eca
2,What are the types of biases that can arise wi...,**Few‑shot prompting can introduce several sys...,[page_content='Zhao et al. (2021) investigated...,None,The biases that can arise with few-shot prompt...,False,True,False,True,0.937658,ff7c3cda-9bed-44af-bffd-a6f5290ede67,019daba3-0c79-7781-974a-d9e36379ae12
